# Experiment 2: Candidate-Elimination Algorithm
**Course:** ITA06 - Machine Learning (Lab)

## Aim
For training examples stored in a **.CSV** file, implement the **Candidate-Elimination** algorithm and output the description of the set of all hypotheses consistent with the training examples (the *version space*).

## Algorithm
* `S` = most specific boundary, `G` = most general boundary (initially `<?, ?, ..., ?>`).
* **Positive example:** remove from `G` any hypothesis inconsistent with it; generalise `S` minimally so it covers the example.
* **Negative example:** for every hypothesis in `G` that wrongly covers it, replace it by its minimal specialisations that are still more general than `S`; keep only maximally general ones.
* The version space lies between `S` and `G`.

In [1]:
import pandas as pd
import itertools

rows = [
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same',   'Yes'],
    ['Sunny', 'Warm', 'High',   'Strong', 'Warm', 'Same',   'Yes'],
    ['Rainy', 'Cold', 'High',   'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High',   'Strong', 'Cool', 'Change', 'Yes'],
]
cols = ['Sky', 'AirTemp', 'Humidity', 'Wind', 'Water', 'Forecast', 'EnjoySport']
pd.DataFrame(rows, columns=cols).to_csv('enjoysport.csv', index=False)

data = pd.read_csv('enjoysport.csv')       # training data from the CSV file
data

,Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
0,Sunny,Warm,Normal,Strong,Warm,Same,Yes
1,Sunny,Warm,High,Strong,Warm,Same,Yes
2,Rainy,Cold,High,Strong,Warm,Change,No
3,Sunny,Warm,High,Strong,Cool,Change,Yes


In [2]:
def covers(h, x):
    """True if hypothesis h classifies instance x as positive."""
    return all(hv == '?' or (hv != '0' and hv == xv) for hv, xv in zip(h, x))

def more_general_or_equal(h1, h2):
    return all(a == '?' or a == b for a, b in zip(h1, h2))

def candidate_elimination(df):
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    n = X.shape[1]
    S = ['0'] * n
    G = [['?'] * n]
    print('S0 =', S)
    print('G0 =', G, '\n')

    for i, (x, label) in enumerate(zip(X, y), start=1):
        x = list(x)
        if label == 'Yes':
            G = [g for g in G if covers(g, x)]              # drop inconsistent g
            for j in range(n):                                # generalise S
                if S[j] == '0':
                    S[j] = x[j]
                elif S[j] != x[j]:
                    S[j] = '?'
        else:
            newG = []
            for g in G:
                if not covers(g, x):
                    newG.append(g)
                else:                                         # specialise g
                    for j in range(n):
                        if g[j] == '?' and S[j] not in ('?', '0') and S[j] != x[j]:
                            h = g.copy()
                            h[j] = S[j]
                            newG.append(h)
            uniq = []
            for g in newG:                                    # remove duplicates
                if g not in uniq:
                    uniq.append(g)
            G = [g for g in uniq                              # keep maximally general
                 if not any(g2 != g and more_general_or_equal(g2, g) for g2 in uniq)]
        print(f'Example {i}: {x} -> {label}')
        print(f'   S{i} =', S)
        print(f'   G{i} =', G, '\n')
    return S, G

S_final, G_final = candidate_elimination(data)

S0 = ['0', '0', '0', '0', '0', '0']
G0 = [['?', '?', '?', '?', '?', '?']] 

Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Yes
   S1 = ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
   G1 = [['?', '?', '?', '?', '?', '?']] 

Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Yes
   S2 = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
   G2 = [['?', '?', '?', '?', '?', '?']] 

Example 3: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> No
   S3 = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
   G3 = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']] 

Example 4: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change'] -> Yes
   S4 = ['Sunny', 'Warm', '?', 'Strong', '?', '?']
   G4 = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']] 



In [3]:
print('Final specific boundary S:')
print('  ', S_final)
print('Final general boundary G:')
for g in G_final:
    print('  ', g)

# Enumerate the complete version space (all hypotheses between S and G)
choices = [[s, '?'] if s != '?' else ['?'] for s in S_final]
version_space = []
for h in itertools.product(*choices):
    h = list(h)
    if any(more_general_or_equal(g, h) for g in G_final):
        version_space.append(h)

print(f'\nVersion space contains {len(version_space)} hypotheses:')
for h in version_space:
    print('  ', h)

Final specific boundary S:
   ['Sunny', 'Warm', '?', 'Strong', '?', '?']
Final general boundary G:
   ['Sunny', '?', '?', '?', '?', '?']
   ['?', 'Warm', '?', '?', '?', '?']

Version space contains 6 hypotheses:
   ['Sunny', 'Warm', '?', 'Strong', '?', '?']
   ['Sunny', 'Warm', '?', '?', '?', '?']
   ['Sunny', '?', '?', 'Strong', '?', '?']
   ['Sunny', '?', '?', '?', '?', '?']
   ['?', 'Warm', '?', 'Strong', '?', '?']
   ['?', 'Warm', '?', '?', '?', '?']


## Result
The algorithm outputs the boundary sets `S = {<Sunny, Warm, ?, Strong, ?, ?>}` and `G = {<Sunny,?,?,?,?,?>, <?,Warm,?,?,?,?>}`; every hypothesis lying between them is consistent with all four training examples.